# SVD, PCA and Matrix Decompositions

Notebook for **Chapter 9** of the *math* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/math/ch09.html

Tip: change a number, predict the output, then run the cell again.

## 9.1 Why Decompose a Matrix, and What the SVD Says

In [ ]:
import numpy as np
A = np.array([[3.0, 0.0], [4.0, 5.0]])
U, s, Vt = np.linalg.svd(A)
print("U =\n", np.round(U, 4))
print("s =", np.round(s, 4))
print("Vt =\n", np.round(Vt, 4))
print("rebuilt =\n", np.round(U @ np.diag(s) @ Vt, 4))
print("sqrt(45), sqrt(5) =", np.round(np.sqrt([45, 5]), 4))

U =
 [[-0.3162 -0.9487]
 [-0.9487  0.3162]]
s = [6.7082 2.2361]
Vt =
 [[-0.7071 -0.7071]
 [-0.7071  0.7071]]
rebuilt =
 [[3. 0.]
 [4. 5.]]
sqrt(45), sqrt(5) = [6.7082 2.2361]


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
A = rng.integers(-3, 4, size=(5, 3)).astype(float)
U, s, Vt = np.linalg.svd(A)                          # full
print("full:", U.shape, s.shape, Vt.shape)
U2, s2, Vt2 = np.linalg.svd(A, full_matrices=False)  # thin
print("thin:", U2.shape, s2.shape, Vt2.shape)
print("same singular values:", np.allclose(s, s2))
print("thin rebuild ok:", np.allclose(U2 @ np.diag(s2) @ Vt2, A))
S_full = np.zeros((5, 3)); S_full[:3, :3] = np.diag(s)
print("full rebuild ok:", np.allclose(U @ S_full @ Vt, A))

full: (5, 5) (3,) (3, 3)
thin: (5, 3) (3,) (3, 3)
same singular values: True
thin rebuild ok: True
full rebuild ok: True


In [ ]:
import numpy as np
A = np.array([[3.0, 2.0, 2.0], [2.0, 3.0, -2.0]])
s = np.linalg.svd(A, compute_uv=False)
ev_ata = np.linalg.eigvalsh(A.T @ A)[::-1]   # 3 eigenvalues, largest first
ev_aat = np.linalg.eigvalsh(A @ A.T)[::-1]   # 2 eigenvalues
print("singular values      :", np.round(s, 4))
print("eig of A^T A (3x3)   :", np.round(ev_ata, 4))
print("eig of A A^T (2x2)   :", np.round(ev_aat, 4))
print("sqrt of eig of AAt   :", np.round(np.sqrt(ev_aat), 4))

singular values      : [5. 3.]
eig of A^T A (3x3)   : [25.  9.  0.]
eig of A A^T (2x2)   : [25.  9.]
sqrt of eig of AAt   : [5. 3.]


In [ ]:
import numpy as np
A = np.array([[3.0, 0.0], [4.0, 5.0]])
U, s, Vt = np.linalg.svd(A)
V = Vt.T
for i in range(2):
    print(f"A v{i+1} = {np.round(A @ V[:, i], 4)}  and  s{i+1} u{i+1} = {np.round(s[i] * U[:, i], 4)}")
# longest and shortest stretch over many unit vectors
t = np.linspace(0, 2 * np.pi, 100000)
lengths = np.linalg.norm(A @ np.vstack([np.cos(t), np.sin(t)]), axis=0)
print("max stretch %.4f, min stretch %.4f" % (lengths.max(), lengths.min()))

A v1 = [-2.1213 -6.364 ]  and  s1 u1 = [-2.1213 -6.364 ]
A v2 = [-2.1213  0.7071]  and  s2 u2 = [-2.1213  0.7071]
max stretch 6.7082, min stretch 2.2361


## 9.2 Computing a 2×2 SVD by Hand, and Rank

In [ ]:
import numpy as np
# third column = first + second, so the rank is 2, not 3
A = np.array([[1.0, 2.0, 3.0], [2.0, 1.0, 3.0], [0.0, 1.0, 1.0], [1.0, 1.0, 2.0]])
s = np.linalg.svd(A, compute_uv=False)
print("singular values:", s)
print("rank by matrix_rank:", np.linalg.matrix_rank(A))
print("count s > 1e-10:", int(np.sum(s > 1e-10)))
B = A + 1e-9 * np.random.default_rng(0).normal(size=A.shape)   # tiny noise
sb = np.linalg.svd(B, compute_uv=False)
print("noisy smallest s:", sb[-1])
print("rank, default tolerance:", np.linalg.matrix_rank(B))
print("rank, tol=1e-6        :", np.linalg.matrix_rank(B, tol=1e-6))

singular values: [5.87560309e+00 1.21543752e+00 1.41392373e-16]
rank by matrix_rank: 2
count s > 1e-10: 2
noisy smallest s: 1.690289226305735e-09
rank, default tolerance: 3
rank, tol=1e-6        : 2


## 9.3 Low-Rank Approximation and the Eckart-Young Theorem

In [ ]:
import numpy as np
A = np.array([[3.0, 2.0, 2.0], [2.0, 3.0, -2.0]])
U, s, Vt = np.linalg.svd(A)
A1 = s[0] * np.outer(U[:, 0], Vt[0])          # best rank-1 matrix
print("A1 =\n", np.round(A1, 4))
E_ = A - A1
print("Frobenius error :", np.round(np.linalg.norm(E_), 4), "(expected s2 = 3)")
print("spectral error  :", np.round(np.linalg.norm(E_, 2), 4))
# a different rank-1 matrix is never better
rng = np.random.default_rng(1)
best = min(np.linalg.norm(A - np.outer(rng.normal(size=2), rng.normal(size=3))) for _ in range(20000))
print("best of 20000 random rank-1 tries:", np.round(best, 3))

A1 =
 [[2.5 2.5 0. ]
 [2.5 2.5 0. ]]
Frobenius error : 3.0 (expected s2 = 3)
spectral error  : 3.0
best of 20000 random rank-1 tries: 3.392


In [ ]:
import numpy as np
# Eckart-Young check on a random matrix: error = sqrt of the sum of discarded s^2
rng = np.random.default_rng(5)
A = rng.normal(size=(8, 6))
U, s, Vt = np.linalg.svd(A, full_matrices=False)
for k in [1, 2, 3, 4]:
    Ak = (U[:, :k] * s[:k]) @ Vt[:k]
    print(k, "fro error %.4f" % np.linalg.norm(A - Ak), "formula %.4f" % np.sqrt(np.sum(s[k:]**2)),
          "| spectral %.4f" % np.linalg.norm(A - Ak, 2), "s[k] %.4f" % s[k])

1 fro error 5.1975 formula 5.1975 | spectral 3.6255 s[k] 3.6255
2 fro error 3.7242 formula 3.7242 | spectral 2.8298 s[k] 2.8298
3 fro error 2.4212 formula 2.4212 | spectral 2.0421 s[k] 2.0421
4 fro error 1.3008 formula 1.3008 | spectral 1.2807 s[k] 1.2807


In [ ]:
import numpy as np
def make_image(n=64):
    y, x = np.mgrid[0:n, 0:n] / (n - 1)
    img = 0.5 * x + 0.3 * y
    img += 0.4 * (((x - 0.65)**2 + (y - 0.4)**2) < 0.04)
    img += 0.25 * (np.sin(8 * np.pi * x) > 0) * (y > 0.7)
    img += 0.2 * ((x > 0.1) & (x < 0.3) & (y > 0.1) & (y < 0.35))
    img = img + 0.02 * np.random.default_rng(0).normal(size=(n, n))
    return img
img = make_image()
U, s, Vt = np.linalg.svd(img, full_matrices=False)
print("original numbers stored:", img.size)
for k in [1, 2, 5, 10, 20]:
    approx = (U[:, :k] * s[:k]) @ Vt[:k]
    stored = k * (64 + 64 + 1)
    rel = np.linalg.norm(img - approx) / np.linalg.norm(img)
    check = np.sqrt(np.sum(s[k:]**2)) / np.linalg.norm(img)
    print(f"k={k:2d} stored={stored:5d} ratio={img.size/stored:5.1f} rel_error={rel:.4f} check={check:.4f}")

original numbers stored: 4096
k= 1 stored=  129 ratio= 31.8 rel_error=0.2076 check=0.2076
k= 2 stored=  258 ratio= 15.9 rel_error=0.1322 check=0.1322
k= 5 stored=  645 ratio=  6.4 rel_error=0.0672 check=0.0672
k=10 stored= 1290 ratio=  3.2 rel_error=0.0391 check=0.0391
k=20 stored= 2580 ratio=  1.6 rel_error=0.0231 check=0.0231


## 9.4 The Pseudo-Inverse and Least Squares

In [ ]:
import numpy as np
# A x in terms of SVD: rank-1 pieces. Dense example of the pseudo-inverse of a diagonal S
S = np.diag([3.0, 2.0, 0.0])
print(np.linalg.pinv(S))
A = np.array([[1.0, 2.0], [2.0, 4.0]])             # rank 1
print("pinv(A) =\n", np.round(np.linalg.pinv(A), 4))
print("A pinv(A) A = A:", np.allclose(A @ np.linalg.pinv(A) @ A, A))
print("min-norm solution of A x = (3, 6):", np.round(np.linalg.pinv(A) @ np.array([3.0, 6.0]), 4))

[[0.33333333 0.         0.        ]
 [0.         0.5        0.        ]
 [0.         0.         0.        ]]
pinv(A) =
 [[0.04 0.08]
 [0.08 0.16]]
A pinv(A) A = A: True
min-norm solution of A x = (3, 6): [0.6 1.2]


In [ ]:
import numpy as np
x = np.array([1.0, 2.0, 3.0, 4.0]); y = np.array([2.0, 3.0, 5.0, 6.0])
X = np.column_stack([np.ones(4), x])
U, s, Vt = np.linalg.svd(X, full_matrices=False)
beta = Vt.T @ np.diag(1 / s) @ U.T @ y          # pseudo-inverse built by hand
print("beta via SVD  :", np.round(beta, 4))
print("beta via pinv :", np.round(np.linalg.pinv(X) @ y, 4))
print("beta via lstsq:", np.round(np.linalg.lstsq(X, y, rcond=None)[0], 4))
# rank-deficient: duplicated column, infinitely many solutions, pinv picks the shortest
Xd = np.column_stack([np.ones(4), x, x])
b = np.linalg.pinv(Xd) @ y
print("pinv on duplicated column:", np.round(b, 4), "norm", np.round(np.linalg.norm(b), 4))
print("fit still perfect as before:", np.allclose(Xd @ b, X @ beta))

beta via SVD  : [0.5 1.4]
beta via pinv : [0.5 1.4]
beta via lstsq: [0.5 1.4]
pinv on duplicated column: [0.5 0.7 0.7] norm 1.1091
fit still perfect as before: True


## 9.5 PCA: Principal Component Analysis from Scratch

In [ ]:
import numpy as np
X = np.array([[1, 2], [2, 3], [3, 5], [4, 4], [5, 6]], float)
Xc = X - X.mean(axis=0)                       # centre: subtract (3, 4)
C = Xc.T @ Xc / (len(X) - 1)                  # covariance matrix
vals, vecs = np.linalg.eigh(C)                # eigh sorts ascending
v1 = vecs[:, 1] * np.sign(vecs[0, 1])         # PC1, sign fixed so it points up-right
print("covariance =\n", C)
print("eigenvalues:", np.round(vals[::-1], 4))
print("PC1 direction:", np.round(v1, 4))
print("PC1 scores:", np.round(Xc @ v1, 4))
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
print("singular values:", np.round(s, 4), "| sqrt(19) =", np.round(np.sqrt(19), 4))
print("s^2/(n-1):", np.round(s**2 / 4, 4))
print("explained ratio:", np.round(vals[::-1] / vals.sum(), 4))

covariance =
 [[2.5  2.25]
 [2.25 2.5 ]]
eigenvalues: [4.75 0.25]
PC1 direction: [0.7071 0.7071]
PC1 scores: [-2.8284 -1.4142  0.7071  0.7071  2.8284]
singular values: [4.3589 1.    ] | sqrt(19) = 4.3589
s^2/(n-1): [4.75 0.25]
explained ratio: [0.95 0.05]


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
rng = np.random.default_rng(42)
n = 200
z = rng.normal(size=(n, 2)) * [3.0, 1.0]            # two real directions
W = np.array([[1.0, 0.5, 0.2, 0.0], [0.0, 0.5, 1.0, 1.0]])
X = z @ W + 0.1 * rng.normal(size=(n, 4)) + [10, 20, 30, 40]   # 4 features, about 2 dims
def my_pca(X, k):
    Xc = X - X.mean(axis=0)
    U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = s**2 / (len(X) - 1)
    return Xc @ Vt[:k].T, Vt[:k], var
scores, comps, var = my_pca(X, 2)
sk = PCA(n_components=2).fit(X)
print("my variances     :", np.round(var, 4))
print("sklearn variances:", np.round(sk.explained_variance_, 4))
print("my ratio         :", np.round(var / var.sum(), 4))
print("sklearn ratio    :", np.round(sk.explained_variance_ratio_, 4))
same = np.allclose(np.abs(comps), np.abs(sk.components_), atol=1e-6)
print("components equal up to sign:", same)
print("scores equal up to sign   :", np.allclose(np.abs(scores), np.abs(sk.transform(X)), atol=1e-6))
print("cumulative ratio:", np.round(np.cumsum(var / var.sum()), 4))

my variances     : [1.17569e+01 1.71120e+00 1.03000e-02 7.30000e-03]
sklearn variances: [11.7569  1.7112]
my ratio         : [8.718e-01 1.269e-01 8.000e-04 5.000e-04]
sklearn ratio    : [0.8718 0.1269]
components equal up to sign: True
scores equal up to sign   : True
cumulative ratio: [0.8718 0.9987 0.9995 1.    ]


In [ ]:
import numpy as np
rng = np.random.default_rng(42)
n = 200
z = rng.normal(size=(n, 2)) * [3.0, 1.0]
W = np.array([[1.0, 0.5, 0.2, 0.0], [0.0, 0.5, 1.0, 1.0]])
X = z @ W + 0.1 * rng.normal(size=(n, 4)) + [10, 20, 30, 40]
Xc = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
var = s**2 / (n - 1)
for k in [1, 2, 3]:
    Xhat = (Xc @ Vt[:k].T) @ Vt[:k]                 # project down, then back up
    mse = np.mean(np.sum((Xc - Xhat) ** 2, axis=1))  # average squared distance per row
    print(f"k={k}: reconstruction error {mse:.4f}   dropped variance*(n-1)/n {var[k:].sum() * (n - 1) / n:.4f}")

k=1: reconstruction error 1.7201   dropped variance*(n-1)/n 1.7201
k=2: reconstruction error 0.0175   dropped variance*(n-1)/n 0.0175
k=3: reconstruction error 0.0073   dropped variance*(n-1)/n 0.0073


In [ ]:
import numpy as np
# SVD of a centered data matrix gives the same PCA as the covariance matrix, and the pitfall: forgetting to center
rng = np.random.default_rng(0)
X = rng.normal(size=(100, 2)) @ np.array([[2.0, 0.0], [0.0, 0.5]]) + [50, 50]
s_raw = np.linalg.svd(X, compute_uv=False)
s_ctr = np.linalg.svd(X - X.mean(0), compute_uv=False)
print("uncentered singular values:", np.round(s_raw, 2))
print("centered singular values  :", np.round(s_ctr, 2))
print("centered s^2/(n-1)        :", np.round(s_ctr**2 / 99, 3), "(variances along the axes)")

uncentered singular values: [706.61  13.68]
centered singular values  : [19.41  4.68]
centered s^2/(n-1)        : [3.804 0.221] (variances along the axes)


In [ ]:
import numpy as np
rng = np.random.default_rng(7)
Sigma = np.array([[4.0, 1.8], [1.8, 1.0]])
X = rng.multivariate_normal([0, 0], Sigma, size=5000)
Xc = X - X.mean(axis=0)
C = np.cov(Xc, rowvar=False)
vals, vecs = np.linalg.eigh(C)
Xw = Xc @ vecs / np.sqrt(vals)                   # PCA whitening
print("cov before:\n", np.round(C, 3))
print("cov after whitening:\n", np.round(np.cov(Xw, rowvar=False), 3))

cov before:
 [[3.986 1.777]
 [1.777 0.979]]
cov after whitening:
 [[ 1. -0.]
 [-0.  1.]]


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
rng = np.random.default_rng(3)
n = 300
height_cm = rng.normal(170, 10, n)
income = 30000 + 500 * (height_cm - 170) + rng.normal(0, 8000, n)   # dollars
X = np.column_stack([height_cm, income])
p1 = PCA(2).fit(X)
r = p1.explained_variance_ratio_; w = np.abs(p1.components_[0])
print(f"raw units    : PC1 explains {r[0]:.6f}, loadings height {w[0]:.4f}, income {w[1]:.4f}")
Z = (X - X.mean(0)) / X.std(0, ddof=1)
p2 = PCA(2).fit(Z)
r = p2.explained_variance_ratio_; w = np.abs(p2.components_[0])
print(f"standardised : PC1 explains {r[0]:.4f}, loadings height {w[0]:.4f}, income {w[1]:.4f}")

raw units    : PC1 explains 0.999999, loadings height 0.0006, income 1.0000
standardised : PC1 explains 0.7943, loadings height 0.7071, income 0.7071


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
# outlier pitfall: one wild point rotates PC1
rng = np.random.default_rng(0)
X = rng.normal(size=(100, 2)) * [3.0, 0.5]           # long along x
print("PC1 without outlier:", np.round(np.abs(PCA(1).fit(X).components_[0]), 3))
X2 = np.vstack([X, [0.0, 40.0]])                      # one point far up the y axis
print("PC1 with outlier   :", np.round(np.abs(PCA(1).fit(X2).components_[0]), 3))

PC1 without outlier: [1.    0.023]
PC1 with outlier   : [0.036 0.999]


## 9.6 Latent Semantic Analysis and Recommender Systems

In [ ]:
import numpy as np
# rows = terms, columns = documents d1..d4
terms = ["cat", "dog", "pet", "stock", "market"]
A = np.array([[2, 1, 0, 0],    # cat
              [1, 2, 0, 0],    # dog
              [1, 1, 0, 1],    # pet
              [0, 0, 2, 1],    # stock
              [0, 0, 1, 2]], float)   # market
U, s, Vt = np.linalg.svd(A, full_matrices=False)
print("singular values:", np.round(s, 3))
k = 2
docs = (np.diag(s[:k]) @ Vt[:k]).T            # each document as a point in 2 concept dimensions
print("doc coordinates:\n", np.round(docs, 3))
def cos(a, b): return a @ b / np.linalg.norm(a) / np.linalg.norm(b)
print("cos(d1,d2) raw:", np.round(cos(A[:, 0], A[:, 1]), 3), " latent:", np.round(cos(docs[0], docs[1]), 3))
print("cos(d2,d4) raw:", np.round(cos(A[:, 1], A[:, 3]), 3), " latent:", np.round(cos(docs[1], docs[3]), 3))
print("cos(d3,d4) raw:", np.round(cos(A[:, 2], A[:, 3]), 3), " latent:", np.round(cos(docs[2], docs[3]), 3))
print("term loadings on concept 1:", [(t, round(float(v), 2)) for t, v in zip(terms, U[:, 0])])
print("term loadings on concept 2:", [(t, round(float(v), 2)) for t, v in zip(terms, U[:, 1])])

singular values: [3.409 3.    1.173 1.   ]
doc coordinates:
 [[-2.143  0.949]
 [-2.143  0.949]
 [-0.807 -1.897]
 [-1.336 -1.897]]
cos(d1,d2) raw: 0.833  latent: 1.0
cos(d2,d4) raw: 0.167  latent: 0.196
cos(d3,d4) raw: 0.73  latent: 0.978
term loadings on concept 1: [('cat', -0.55), ('dog', -0.55), ('pet', -0.48), ('stock', -0.25), ('market', -0.3)]
term loadings on concept 2: [('cat', 0.32), ('dog', 0.32), ('pet', 0.0), ('stock', -0.63), ('market', -0.63)]


In [ ]:
import pandas as pd
ratings = pd.DataFrame({
    "user":  ["Ana", "Ana", "Ana", "Ben", "Ben", "Cy", "Cy", "Cy"],
    "movie": ["Heat", "Alien", "Titanic", "Heat", "Titanic", "Alien", "Titanic", "Heat"],
    "stars": [5, 4, 1, 4, 2, 5, 1, 5]})
M = ratings.pivot(index="user", columns="movie", values="stars")
print(M)
print("known entries:", int(M.notna().sum().sum()), "of", M.size)
print("sparsity:", round(1 - M.notna().sum().sum() / M.size, 3))

movie  Alien  Heat  Titanic
user                       
Ana      4.0   5.0      1.0
Ben      NaN   4.0      2.0
Cy       5.0   5.0      1.0
known entries: 8 of 9
sparsity: 0.111


In [ ]:
import numpy as np
R = np.array([[5, 4, np.nan, 1, 1], [4, np.nan, 5, 1, 2], [5, 5, 4, np.nan, 1],
              [1, 1, np.nan, 5, 4], [2, np.nan, 1, 4, 5], [1, 2, 1, 5, np.nan]])
mask = ~np.isnan(R)
k, lam = 2, 0.1
rng = np.random.default_rng(0)
P = rng.normal(0, 0.5, (6, k))     # user factors
Q = rng.normal(0, 0.5, (5, k))     # movie factors
for sweep in range(25):
    for u in range(6):             # fix Q, solve a small ridge regression for each user
        Qi = Q[mask[u]]
        P[u] = np.linalg.solve(Qi.T @ Qi + lam * np.eye(k), Qi.T @ R[u, mask[u]])
    for i in range(5):             # fix P, solve for each movie
        Pu = P[mask[:, i]]
        Q[i] = np.linalg.solve(Pu.T @ Pu + lam * np.eye(k), Pu.T @ R[mask[:, i], i])
pred = P @ Q.T
rmse = np.sqrt(np.mean((pred - R)[mask] ** 2))
print("RMSE on known ratings:", np.round(rmse, 3))
print("predicted matrix:\n", np.round(pred, 1))
for u, i in zip(*np.where(~mask)):
    print(f"user {u} movie {i}: predicted {pred[u, i]:.2f}")

RMSE on known ratings: 0.37
predicted matrix:
 [[4.5 4.4 4.3 0.7 1.3]
 [4.6 4.5 4.3 1.2 1.7]
 [4.8 4.6 4.5 0.5 1.1]
 [0.8 1.2 0.3 4.5 4.4]
 [1.8 2.1 1.3 4.5 4.5]
 [1.4 1.8 0.9 4.9 4.9]]
user 0 movie 2: predicted 4.27
user 1 movie 1: predicted 4.51
user 2 movie 3: predicted 0.49
user 3 movie 2: predicted 0.33
user 4 movie 1: predicted 2.15
user 5 movie 4: predicted 4.87


In [ ]:
import numpy as np
# SGD version on the same data: update one known rating at a time
R = np.array([[5, 4, np.nan, 1, 1], [4, np.nan, 5, 1, 2], [5, 5, 4, np.nan, 1],
              [1, 1, np.nan, 5, 4], [2, np.nan, 1, 4, 5], [1, 2, 1, 5, np.nan]])
rows, cols = np.where(~np.isnan(R))
rng = np.random.default_rng(0)
k, lr, lam = 2, 0.05, 0.02
P = rng.normal(0, 0.3, (6, k)); Q = rng.normal(0, 0.3, (5, k))
for epoch in range(300):
    for t in rng.permutation(len(rows)):
        u, i = rows[t], cols[t]
        err = R[u, i] - P[u] @ Q[i]
        pu = P[u].copy()
        P[u] += lr * (err * Q[i] - lam * P[u])
        Q[i] += lr * (err * pu - lam * Q[i])
pred = P @ Q.T
print("RMSE known:", np.round(np.sqrt(np.mean((pred - R)[~np.isnan(R)] ** 2)), 3))
print("user 0 movie 2 predicted:", np.round(pred[0, 2], 2))
print("user 3 movie 2 predicted:", np.round(pred[3, 2], 2))

RMSE known: 0.374
user 0 movie 2 predicted: 4.28
user 3 movie 2 predicted: 0.37


In [ ]:
import numpy as np
# hold out one KNOWN rating and see whether the model recovers it
R = np.array([[5, 4, np.nan, 1, 1], [4, np.nan, 5, 1, 2], [5, 5, 4, np.nan, 1],
              [1, 1, np.nan, 5, 4], [2, np.nan, 1, 4, 5], [1, 2, 1, 5, np.nan]])
true_value = R[0, 1]; R[0, 1] = np.nan          # hide user 0's rating of movie 1 (it was 4)
mask = ~np.isnan(R)
def als(R, mask, k=2, lam=0.1, sweeps=25, seed=0):
    rng = np.random.default_rng(seed)
    P = rng.normal(0, 0.5, (R.shape[0], k)); Q = rng.normal(0, 0.5, (R.shape[1], k))
    for _ in range(sweeps):
        for u in range(R.shape[0]):
            Qi = Q[mask[u]]; P[u] = np.linalg.solve(Qi.T @ Qi + lam*np.eye(k), Qi.T @ R[u, mask[u]])
        for i in range(R.shape[1]):
            Pu = P[mask[:, i]]; Q[i] = np.linalg.solve(Pu.T @ Pu + lam*np.eye(k), Pu.T @ R[mask[:, i], i])
    return P @ Q.T
for lam in [0.1, 0.5, 1.0, 2.0]:
    pred = als(R, mask, lam=lam)
    print(f"lambda={lam}: predicted {pred[0, 1]:.2f} (true {true_value:.0f})")
print("global-mean guess:", np.round(np.nanmean(R), 2))

lambda=0.1: predicted 5.27 (true 4)
lambda=0.5: predicted 4.43 (true 4)
lambda=1.0: predicted 3.71 (true 4)
lambda=2.0: predicted 2.76 (true 4)
global-mean guess: 2.87


## 9.7 Other Decompositions: LU, QR, Cholesky, NMF and Fast SVD

In [ ]:
import numpy as np
from scipy.linalg import lu
def lu_no_pivot(A):
    n = len(A); L = np.eye(n); U = A.astype(float).copy()
    for j in range(n - 1):
        for i in range(j + 1, n):
            L[i, j] = U[i, j] / U[j, j]          # the multiplier
            U[i] -= L[i, j] * U[j]               # row_i = row_i - multiplier * row_j
    return L, U
A = np.array([[2.0, 1.0, 1.0], [4.0, 3.0, 3.0], [8.0, 7.0, 9.0]])
L, U = lu_no_pivot(A)
print("L =\n", L)
print("U =\n", U)
print("L U = A:", np.allclose(L @ U, A), "| det = product of U diagonal:", np.prod(np.diag(U)))
P, L2, U2 = lu(A)                                  # scipy swaps rows for stability: A = P L U
print("scipy: P L U = A:", np.allclose(P @ L2 @ U2, A))

L =
 [[1. 0. 0.]
 [2. 1. 0.]
 [4. 3. 1.]]
U =
 [[2. 1. 1.]
 [0. 1. 1.]
 [0. 0. 2.]]
L U = A: True | det = product of U diagonal: 4.0
scipy: P L U = A: True


In [ ]:
import numpy as np
A = np.array([[1.0, 1.0], [1.0, 0.0]])
Q, R = np.linalg.qr(A)
print("Q =\n", np.round(Q, 4))
print("R =\n", np.round(R, 4))
print("sqrt(2), 1/sqrt(2):", np.round([np.sqrt(2), 1/np.sqrt(2)], 4))
print("Q^T Q = I:", np.allclose(Q.T @ Q, np.eye(2)), "| QR = A:", np.allclose(Q @ R, A))
# least squares with QR: solve R beta = Q^T y (back substitution)
x = np.array([1.0, 2.0, 3.0, 4.0]); y = np.array([2.0, 3.0, 5.0, 6.0])
X = np.column_stack([np.ones(4), x])
Q, R = np.linalg.qr(X)
beta = np.linalg.solve(R, Q.T @ y)
print("beta via QR:", np.round(beta, 4))

Q =
 [[-0.7071 -0.7071]
 [-0.7071  0.7071]]
R =
 [[-1.4142 -0.7071]
 [ 0.     -0.7071]]
sqrt(2), 1/sqrt(2): [1.4142 0.7071]
Q^T Q = I: True | QR = A: True
beta via QR: [0.5 1.4]


In [ ]:
import numpy as np
A = np.array([[4.0, 2.0], [2.0, 3.0]])
L = np.linalg.cholesky(A)
print("L =\n", np.round(L, 4), "| sqrt(2) =", np.round(np.sqrt(2), 4))
print("L L^T = A:", np.allclose(L @ L.T, A))
b = np.array([6.0, 5.0])
y = np.linalg.solve(L, b)          # forward substitution (L is triangular)
x = np.linalg.solve(L.T, y)        # back substitution
print("y =", np.round(y, 4), " x =", np.round(x, 4))
# sample from a multivariate normal with covariance A
rng = np.random.default_rng(0)
Z = rng.normal(size=(100000, 2))   # independent standard normals
X = Z @ L.T                        # each row is L z
print("sample covariance:\n", np.round(np.cov(X, rowvar=False), 2))
# not positive definite: Cholesky fails
try:
    np.linalg.cholesky(np.array([[1.0, 2.0], [2.0, 1.0]]))
except np.linalg.LinAlgError as e:
    print("failed:", e)

L =
 [[2.     0.    ]
 [1.     1.4142]] | sqrt(2) = 1.4142
L L^T = A: True
y = [3.     1.4142]  x = [1. 1.]
sample covariance:
 [[4.02 1.99]
 [1.99 2.99]]
failed: Matrix is not positive definite


In [ ]:
import numpy as np
A = np.array([[1.0, 1.0], [0.0, 1.0]])           # a shear, not symmetric
print("eigenvalues    :", np.round(np.linalg.eigvals(A).real, 4), "(only one direction survives)")
print("singular values:", np.round(np.linalg.svd(A, compute_uv=False), 4))
B = np.array([[2.0, 1.0], [1.0, 2.0]])           # symmetric, positive: they agree
print("sym eig:", np.linalg.eigvalsh(B)[::-1], "| sym svd:", np.linalg.svd(B, compute_uv=False))
C = np.array([[1.0, 0.0], [0.0, -3.0]])          # symmetric with a negative eigenvalue
print("eig:", np.linalg.eigvalsh(C), "| singular values:", np.linalg.svd(C, compute_uv=False))

eigenvalues    : [1. 1.] (only one direction survives)
singular values: [1.618 0.618]
sym eig: [3. 1.] | sym svd: [3. 1.]
eig: [-3.  1.] | singular values: [3. 1.]


In [ ]:
import numpy as np
from sklearn.decomposition import NMF
# non-negative ratings-like matrix with two clear groups
V = np.array([[5, 4, 0, 1], [4, 5, 1, 0], [0, 1, 5, 4], [1, 0, 4, 5]], float)
m = NMF(n_components=2, init="nndsvd", max_iter=500, random_state=0)
W = m.fit_transform(V); H = m.components_
print("W (users x parts) =\n", np.round(W, 2))
print("H (parts x items) =\n", np.round(H, 2))
print("all entries >= 0:", bool((W >= 0).all() and (H >= 0).all()))
print("reconstruction error:", np.round(np.linalg.norm(V - W @ H), 3))

W (users x parts) =
 [[0.21 2.37]
 [0.21 2.37]
 [1.93 0.  ]
 [1.93 0.  ]]
H (parts x items) =
 [[0.26 0.26 2.34 2.34]
 [1.88 1.88 0.   0.  ]]
all entries >= 0: True
reconstruction error: 2.0


In [ ]:
import numpy as np
def randomized_svd(A, k, oversample=5, seed=0):
    rng = np.random.default_rng(seed)
    Omega = rng.normal(size=(A.shape[1], k + oversample))   # random test directions
    Y = A @ Omega                                           # sample the column space
    Q, _ = np.linalg.qr(Y)                                  # orthonormal basis for it
    B = Q.T @ A                                             # small (k+p) x n matrix
    Ub, s, Vt = np.linalg.svd(B, full_matrices=False)
    return (Q @ Ub)[:, :k], s[:k], Vt[:k]
rng = np.random.default_rng(1)
A = rng.normal(size=(2000, 5)) @ rng.normal(size=(5, 800)) + 0.01 * rng.normal(size=(2000, 800))
U, s, Vt = randomized_svd(A, 5)
exact = np.linalg.svd(A, compute_uv=False)[:5]
print("randomized:", np.round(s, 3))
print("exact     :", np.round(exact, 3))

randomized: [1341.793 1283.611 1241.731 1193.224 1163.166]
exact     : [1341.798 1283.62  1241.736 1193.238 1163.181]


In [ ]:
import numpy as np
from scipy.sparse import random as sprand
from scipy.sparse.linalg import svds
from sklearn.decomposition import TruncatedSVD
A = sprand(2000, 1000, density=0.005, random_state=0, format="csr")   # 99.5% zeros
print("stored nonzeros:", A.nnz, "of", A.shape[0] * A.shape[1])
exact = np.linalg.svd(A.toarray(), compute_uv=False)[:3]
s = svds(A, k=3, return_singular_vectors=False)
t = TruncatedSVD(n_components=3, n_iter=20, random_state=0).fit(A)
print("dense SVD    :", np.round(exact, 3))
print("svds (Lanczos):", np.round(np.sort(s)[::-1], 3))
print("TruncatedSVD :", np.round(t.singular_values_, 3))

stored nonzeros: 10000 of 2000000
dense SVD    : [4.372 3.677 3.495]
svds (Lanczos): [4.372 3.677 3.495]
TruncatedSVD : [4.372 3.677 3.495]


## 9.8 The Expert Layer: Norms, Conditioning, LoRA and Embeddings

In [ ]:
import numpy as np
A = np.array([[3.0, 0.0], [4.0, 5.0]])
s = np.linalg.svd(A, compute_uv=False)
print("Frobenius :", np.round(np.linalg.norm(A, 'fro'), 4), "= sqrt(sum s^2) =", np.round(np.sqrt(np.sum(s**2)), 4))
print("spectral  :", np.round(np.linalg.norm(A, 2), 4), "= s_max =", np.round(s[0], 4))
print("nuclear   :", np.round(np.linalg.norm(A, 'nuc'), 4), "= sum s =", np.round(s.sum(), 4))
print("condition :", np.round(np.linalg.cond(A), 4), "= s_max/s_min =", np.round(s[0] / s[-1], 4))

Frobenius : 7.0711 = sqrt(sum s^2) = 7.0711
spectral  : 6.7082 = s_max = 6.7082
nuclear   : 8.9443 = sum s = 8.9443
condition : 3.0 = s_max/s_min = 3.0


In [ ]:
import numpy as np
# nearly parallel columns give a huge condition number and unstable solutions
A = np.array([[1.0, 1.0], [1.0, 1.0001]])
print("cond(A) =", np.round(np.linalg.cond(A)))
b1 = np.array([2.0, 2.0001]); b2 = np.array([2.0, 2.0002])   # tiny change in b
print("x for b1:", np.round(np.linalg.solve(A, b1), 4))
print("x for b2:", np.round(np.linalg.solve(A, b2), 4))
# normal equations square the condition number
X = np.vander(np.linspace(0, 1, 20), 8)
print("cond(X)     = %.3g" % np.linalg.cond(X))
print("cond(X^T X) = %.3g" % np.linalg.cond(X.T @ X))

cond(A) = 40002.0
x for b1: [1. 1.]
x for b2: [0. 2.]
cond(X)     = 1.05e+05
cond(X^T X) = 1.11e+10


In [ ]:
import numpy as np
d, r = 4096, 8
full = d * d
lora = d * r + r * d
print("full update params :", full)
print("LoRA rank-8 params :", lora)
print("fraction           : %.4f%%" % (100 * lora / full))
# a low-rank update B A applied to a vector never needs the d x d matrix
rng = np.random.default_rng(0)
d_small = 6
W = rng.normal(size=(d_small, d_small))
B = rng.normal(size=(d_small, 2)); A = rng.normal(size=(2, d_small))
x = rng.normal(size=d_small)
y1 = (W + B @ A) @ x
y2 = W @ x + B @ (A @ x)
print("same output:", np.allclose(y1, y2), "| rank of B A:", np.linalg.matrix_rank(B @ A))

full update params : 16777216
LoRA rank-8 params : 65536
fraction           : 0.3906%
same output: True | rank of B A: 2


## Chapter recap

## Practice set

Problem 9.9.1 A = diag(5, 0, 2). What are its singular values, its rank and its Frobenius norm?

In [ ]:
import numpy as np
A = np.array([[1.0, 1.0], [1.0, 1.0]])
print(np.round(np.linalg.svd(A, compute_uv=False), 4))
print(np.linalg.matrix_rank(A))

[2. 0.]
1


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
X = rng.normal(size=(100, 3)) @ np.array([[3.0, 1.0, 0.5], [0.0, 1.0, 2.0], [0.0, 0.0, 0.2]])
Xc = X - X.mean(axis=0)
U, s, V = np.linalg.svd(Xc, full_matrices=False)
scores = Xc @ V[:, :2]                       # intended: scores on the first 2 components
print(scores.shape)
good = Xc @ V.T[:, :2]
print("variance of bug scores :", np.round(scores.var(axis=0, ddof=1), 3))
print("variance of right ones :", np.round(good.var(axis=0, ddof=1), 3))

(100, 2)
variance of bug scores : [5.764 8.764]
variance of right ones : [13.057  4.469]


In [ ]:
import numpy as np
A = np.array([[1.0, 2.0], [3.0, 4.0]])
s = np.linalg.svd(A, compute_uv=False)
print(np.round(s, 4), np.round(s.prod(), 4), np.round((s**2).sum(), 4))
print(np.round(np.sqrt(15 + np.array([1, -1]) * np.sqrt(221)), 4))
B = np.array([[3.0, 1.0], [4.0, 0.0]])
Q, R = np.linalg.qr(B)
print(np.round(np.abs(Q), 4)); print(np.round(np.abs(R), 4))
print(np.round(np.cond if False else np.linalg.cond(np.diag([100.0, 0.5])), 4))
print(81920 / 4194304, 16 * (1024 + 4096), 1024 * 4096)
print(np.round(np.linalg.eigvalsh(np.array([[5.0, 2.0], [2.0, 2.0]])), 4))
print(np.linalg.cholesky(np.array([[9.0, 3.0], [3.0, 5.0]])))

[5.465 0.366] 2.0 30.0
[5.465 0.366]
[[0.6 0.8]
 [0.8 0.6]]
[[5.  0.6]
 [0.  0.8]]
200.0
0.01953125 81920 4194304
[1. 6.]
[[3. 0.]
 [1. 2.]]


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/math/ch09.html